In [1]:
import os

# used for configuring biogeme use of GPU, unused
# os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# Destination-choice + stay/move model via Larch

In [2]:
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import larch as lx
import numpy as np
import pandas as pd
from larch import PX

sys.path.insert(0, os.path.abspath(".."))
from lib import model_spec as lm
from lib import modeling_util as lut
from lib import io as lio


In [3]:
num_alternatives = 100
unixtime = int(datetime.now(UTC).timestamp())
path = "../data/estdata_50_2018_100.parquet"
data_file = Path(path).stem

### Read data

In [4]:
df_train = lio.read_estdata(
    path,
    num_alternatives,
)
print(df_train.shape)

(1265450, 1862)


/workspace/migration/lib/io.py:61: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["person_id"] = np.arange(len(df))
/workspace/migration/lib/io.py:64: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["ALT_CHOICE"] = 0


In [5]:
# need to have the sentinel values be different so that SAME_CBSA works correctly
df_train["NAME_NUM.ORIG"].min(), df_train["ALT1_CBSA"].min()

(np.int64(-2), np.float32(-1.0))

In [6]:
for c in df_train.columns:
    print(c)

AGE_18_22
ORIGIN_STATE
EDU_HIGH_BUT_NOT_BACHELORS
NAICS_HIGH_ED
TYPE_NUM.ORIG
House vacancy proportion.ORIG
Proportion of people 18-34.ORIG
Total Population.Total Population.SE_A00001_001.ORIG
WHITE
EDU_HAS_DEGREE
NAICS_GROUP_PROP_AGR_EXT.ORIG
EDU_BACHELORS_OR_HIGHER
INDIAN
Proportion of people 65+.ORIG
BLACK
CHILD
CHILD_UNDER_6
NAICS_GROUP_PROP_GOODS_TRADE.ORIG
AGE_OVER_65
AGE_40_49
JAN_AVG_TEMP_C.ORIG
NAICS_GOODS_TRADE
MARRIED_MORE_THAN_YEAR
SINGLE_PARENT
Median travel time.ORIG
CHOSEN
RECENTLY_WIDOWED_OR_DIVORCED
NAICS_GROUP_PROP_HIGH_ED.ORIG
AGE_50_64
IN_COLLEGE
EDU_NO_DEGREE
AGE_30_39
IN_MILITARY
Proportion of people White.ORIG
Proportion of people Indian.ORIG
Proportion of people AAPI.ORIG
LATINO
FOREIGN
WORK2_MAR
NAICS_GROUP_PROP_GOVT.ORIG
Proportion foreign born.ORIG
Proportion of people Black.ORIG
Proportion of households with children.ORIG
AGE_23_29
Median gross rent in thousands of dollars.ORIG
Proportion of people Latino.ORIG
Proportion of people in college.ORIG
NAME_NUM.OR

### Look at collinearity in requested columns

In [7]:
# # stack all alternatives into long format
# frames = []
# for i in range(1, num_alternatives + 1):
#     cols = {
#         f"ALT{i}_{v}": v
#         for v in lm.required_alt_suffixes()
#         if f"ALT{i}_{v}" in df_train.columns
#     }
#     frames.append(df_train[list(cols)].rename(columns=cols))

# long = pd.concat(frames, ignore_index=True)

# # add the transformed versions you actually use in the model
# long["log_DIST"] = np.log(long["DIST"] + 1)
# long["log_TOT_POP"] = np.log(long["TOT_POP"])

# corr = long.corr()

In [8]:
# c = corr.abs()
# pairs = (
#     c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
#     .stack()
#     .sort_values(ascending=False)
# )
# print(pairs[pairs > 0.3])

In [9]:
# orig_vars = [v for v in lm.required_individual_columns() if v in df_train.columns]  # skip any missing
# corr = df_train[orig_vars].corr()

# c = corr.abs()
# pairs = (
#     c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
#     .stack()
#     .sort_values(ascending=False)
# )
# print(pairs[pairs > 0.3].to_string())

### Reshape to long format, build the Larch dataset

`lib.util.build_long_data` builds the long `(person_id, alt)` table shared by this notebook and
`modeling_torch_choice.ipynb`: `alt=0` is staying, `alt=1..num_alternatives` are the move alternatives, with the
stay/move-context values for shared coefficients (e.g. `proportion_same_age_18_34`) written under the
same column name so they tie to one coefficient downstream, and `log_pop_offset` (destination
population on move rows, origin population on the stay row) left as an un-parameterized term. 

The returned `long_df` is already sorted by `(person_id, alt)`; `Dataset.construct.from_idca` takes it
directly (indexed by `(caseid, altid)`) -- no manual reshape into arrays needed, unlike the torch-choice
port.


In [10]:
long_df, STAY_ONLY_TERMS, SHARED_TERMS, MOVE_ONLY_TERMS = lut.build_long_data(
    df_train, num_alternatives
)
varnames = STAY_ONLY_TERMS + SHARED_TERMS + MOVE_ONLY_TERMS

long_df.set_index(["person_id", "alt"], inplace=True)

In [11]:
lut.print_utility_formula(long_df.reset_index()[["person_id", "alt", "choice", "log_pop_offset"] + varnames])

Stay utility:
    Beta(stay)*Variable(stay)
    + Beta(stay_age_18_22)*Variable(stay_age_18_22)
    + Beta(stay_age_23_29)*Variable(stay_age_23_29)
    + Beta(stay_age_30_39)*Variable(stay_age_30_39)
    + Beta(stay_age_40_49)*Variable(stay_age_40_49)
    + Beta(stay_age_50_64)*Variable(stay_age_50_64)
    + Beta(stay_child_under_6)*Variable(stay_child_under_6)
    + Beta(stay_child_6_to_17)*Variable(stay_child_6_to_17)
    + Beta(stay_married_more_than_year)*Variable(stay_married_more_than_year)
    + Beta(stay_married_less_than_year)*Variable(stay_married_less_than_year)
    + Beta(stay_recently_divorced_or_widowed)*Variable(stay_recently_divorced_or_widowed)
    + Beta(stay_2work_mar)*Variable(stay_2work_mar)
    + Beta(stay_single_parent)*Variable(stay_single_parent)
    + Beta(stay_edu_at_least_bachelors)*Variable(stay_edu_at_least_bachelors)
    + Beta(stay_edu_high_no_bachelors)*Variable(stay_edu_high_no_bachelors)
    + Beta(stay_in_college)*Variable(stay_in_college)
    + Beta

In [12]:
ds = lx.Dataset.construct.from_idca(
    long_df[["choice", "log_pop_offset"] + varnames], crack=False
)
ds

<xarray.Dataset> Size: 29GB
Dimensions:                                       (person_id: 1265450, alt: 101)
Coordinates:
  * person_id                                     (person_id) int64 10MB 0 .....
  * alt                                           (alt) int64 808B 0 1 ... 100
Data variables: (12/55)
    choice                                        (person_id, alt) int64 1GB ...
    log_pop_offset                                (person_id, alt) float32 511MB ...
    stay                                          (person_id, alt) float32 511MB ...
    stay_age_18_22                                (person_id, alt) float32 511MB ...
    stay_age_23_29                                (person_id, alt) float32 511MB ...
    stay_age_30_39                                (person_id, alt) float32 511MB ...
    ...                                            ...
    destchoice_metro                              (person_id, alt) float32 511MB ...
    destchoice_same_cbsa_type                     (person_id, alt) float32 511MB ...
    destchoice_vacancy_rate                       (person_id, alt) float32 511MB ...
    destchoice_med_rent_k                         (person_id, alt) float32 511MB ...
    destchoice_med_earnings_10k_no_degree         (person_id, alt) float32 511MB ...
    destchoice_med_earnings_10k_degree            (person_id, alt) float32 511MB ...
Attributes:
    _caseid_:  person_id
    _altid_:   alt

### Setting up the model

One `P(name) * X(name)` term per `varnames` entry via the `PX` shorthand, summed on a plain local
variable and assigned to `m.utility_ca` once at the end -- **not** built with `m.utility_ca += ...` in
a loop, which silently discards everything but the last term (see intro cell). No separate ASC/intercept
term is added -- `stay` in `STAY_ONLY_TERMS` is already an explicit ASC for staying, matching
`fit_intercept=False` in the torch-choice port / Biogeme not adding an implicit ASC of its own.

`log_pop_offset` gets a coefficient too, then `m.lock_value("log_pop_offset", 1)` pins it at exactly 1
(`holdfast`), reproducing Biogeme's bare `log(Variable(...))` calls (and xlogit's `addit=`) without
needing a custom subclass the way the torch-choice port did.


In [13]:
m = lx.Model(ds)
m.title = f"us_mnl_{data_file}_{unixtime}"
m.compute_engine = "numba"

# all alternatives have the same utility function
# stay-specific columns have their values zeroed out for destination alternatives and vice versa
# PX represents a column multiplied by a coefficient that will be estimated
total_utility = PX(varnames[0])
for name in varnames[1:]:
    total_utility = total_utility + PX(name)
total_utility = total_utility + PX("log_pop_offset")
m.utility_ca = total_utility

m.choice_ca_var = "choice"
# all alternatives are available for everyone
# no availability_ca_var needed.

# fix the size term coefficient, it is a constant
m.lock_value("log_pop_offset", 1)

m.ordering = [
    ("Stay", "stay.*"),
    ("Destination-only", "destchoice.*"),
    ("Offset", "log_pop_offset"),
]


In [14]:
# NOTE: nesting showed that mu_move tended to go towards 1, indicating that nesting is not necessary

# # optional cell: turns on the nested structure

# # nested logit: alt=0 (stay) stays a direct root child (== a degenerate nest fixed at 1.0);
# # alts 1..num_alternatives go under a "Move" nest with an estimated logsum coefficient.
# m.graph.new_node(
#     parameter="mu_move",
#     children=list(range(1, num_alternatives + 1)),
#     name="Move",
# )
# m.set_value("mu_move", value=0.5, initvalue=0.5, minimum=0.001, maximum=1.0)

# m.ordering = [
#     ("Stay", "stay.*"),
#     ("Shared", "proportion.*|median_.*|unemp_rate|vacancy_rate"),
#     ("Destination-only", "destchoice.*"),
#     ("Nesting", "mu_.*"),
#     ("Offset", "log_pop_offset"),
# ]
# m.title = f"us_nested_{data_file}_{unixtime}"


### Fitting

In [15]:
print("null log-likelihood:", m.loglike())


null log-likelihood: -4428058.441407948


In [16]:
# Best LL = -585537.6735129762

In [17]:
result = m.maximize_loglike(method="BHHH")
result


┣          loglike: np.float64(-684591.1986120462)
┣                x: destchoice_T34                                  -0.698597
┃                   destchoice_birthstate                            0.280377
┃                   destchoice_logdist                              -0.963625
┃                   destchoice_med_earnings_10k_degree               0.210064
┃                   destchoice_med_earnings_10k_no_degree           -0.029369
┃                   destchoice_med_rent_k                           -0.627074
┃                   destchoice_metro                                -0.248659
┃                   destchoice_same_cbsa_type                        0.285153
┃                   destchoice_samecbsa                              1.473180
┃                   destchoice_samestate                             2.029992
┃                   destchoice_vacancy_rate                          2.358629
┃                   jan_avg_temp_c                                   0.021166
┃                   log_pop_offset                                   1.000000
┃                   median_travel_time                              -0.021670
┃                   proportion_also_latino                           0.563105
┃                   proportion_also_mil                             21.270185
┃                   proportion_college_if_in_college                10.720040
┃                   proportion_foreign_if_foreign                    1.663214
┃                   proportion_hh_with_children_if_have_children     3.659930
┃                   proportion_same_age_18_34                        3.064488
┃                   proportion_same_age_35_64                        1.504393
┃                   proportion_same_age_65_plus                      3.149052
┃                   proportion_same_naics_agr_ext                    5.903729
┃                   proportion_same_naics_goods_trade                1.432951
┃                   proportion_same_naics_govt                       2.331353
┃                   proportion_same_naics_high_ed                    0.473494
┃                   proportion_same_race_aapi                        3.841979
┃                   proportion_same_race_black                       1.778318
┃                   proportion_same_race_indian                      4.575194
┃                   proportion_same_race_white                       2.114276
┃                   stay                                            -6.032984
┃                   stay_2work_mar                                   0.722143
┃                   stay_T34                                         0.246181
┃                   stay_age_18_22                                  -1.832478
┃                   stay_age_23_29                                  -1.851458
┃                   stay_age_30_39                                  -1.493190
┃                   stay_age_40_49                                  -1.062876
┃                   stay_age_50_64                                  -0.475130
┃                   stay_child_6_to_17                               0.408775
┃                   stay_child_under_6                              -0.231934
┃                   stay_edu_at_least_bachelors                      0.115052
┃                   stay_edu_high_no_bachelors                      -0.220446
┃                   stay_foreign                                    -0.137966
┃                   stay_in_college                                 -0.145689
┃                   stay_married_less_than_year                     -0.738288
┃                   stay_married_more_than_year                      0.483388
┃                   stay_med_earnings_10k_degree                     0.316069
┃                   stay_med_earnings_10k_no_degree                  0.273831
┃                   stay_med_rent_k                                 -1.340216
┃                   stay_metro                                       0.164111
┃                   stay_mil                     

In [18]:
m.calculate_parameter_covariance()
m.parameter_summary()


In [19]:
report = lx.Reporter(title=m.title)
report << "# Parameter Summary" << m.parameter_summary()
report << "# Estimation Statistics" << m.estimation_statistics()
report.save(
    f"results/{m.title}.html",
    overwrite=True,
    metadata=m.dumps(),
)
m.save(f"results/{m.title}_spec.yaml")